# Data Exploration

In [15]:
!pip install -q python-terrier ir_datasets pandas

In [16]:
import pyterrier as pt
import pandas as pd

#collecting from trec-covid round5 dataset
dataset_name = "irds:cord19/trec-covid/round5"
dataset = pt.get_dataset(dataset_name)

print(f"Loaded dataset: {dataset_name}")
irds_dataset = dataset.irds_ref()

Loaded dataset: irds:cord19/trec-covid/round5


## Query Exploration

In [17]:
topics_data = []
for query in irds_dataset.queries_iter():
    topics_data.append({
        # 'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })

df_topics = pd.DataFrame(topics_data)
print(f"Total number of queries: {len(df_topics)}")
df_topics.head(3)

Total number of queries: 50


,title,description,narrative
0,coronavirus origin,what is the origin of COVID-19,seeking range of information about the SARS-Co...
1,coronavirus response to weather changes,how does the coronavirus respond to changes in...,seeking range of information about the SARS-Co...
2,coronavirus immunity,will SARS-CoV2 infected people develop immunit...,seeking studies of immunity developed due to i...


## Relevance Judgments Exploration

In [18]:
qrels = dataset.get_qrels()

print(f"Total number of relevance judgments: {len(qrels)}")

print("\nRelevance Score Distribution:")
qrels['label'].value_counts()

Total number of relevance judgments: 23151

Relevance Score Distribution:


label
 0    12239
 2     6677
 1     4233
-1        2
Name: count, dtype: int64

## Document Exploration

In [19]:
doc_iter = irds_dataset.docs_iter()

sample_docs = []
for i, doc in enumerate(doc_iter):
    if i >= 5:
        break
    sample_docs.append({
        'docno': doc.doc_id,
        'title': doc.title,
        'abstract': doc.abstract[:300]
    })

df_docs = pd.DataFrame(sample_docs)
df_docs

,docno,title,abstract
0,ug7v899j,Clinical features of culture-proven Mycoplasma...,OBJECTIVE: This retrospective chart review des...
1,02tnwd4m,Nitric oxide: a pro-inflammatory mediator in l...,Inflammatory diseases of the respiratory tract...
2,ejv2xln0,Surfactant protein-D and pulmonary host defense,Surfactant protein-D (SP-D) participates in th...
3,2b73a28n,Role of endothelin-1 in lung disease,Endothelin-1 (ET-1) is a 21 amino acid peptide...
4,9785vg6d,Gene expression in epithelial cells in respons...,Respiratory syncytial virus (RSV) and pneumoni...


## Indexing

In [20]:
import os

if not os.path.exists("./index/data.properties"):
    indexer = pt.IterDictIndexer("./index")
    
    index_ref = indexer.index(
        ({
            "docno": d.doc_id,
            "text": (d.title or "") + " " + (d.abstract or "")
        }
        for d in irds_dataset.docs_iter())
    )
else:
    index_ref = "./index"

index = pt.IndexFactory.of(index_ref)

In [21]:
#loading the index
index = pt.IndexFactory.of("./cord19_index/data.properties")
#or
#index = pt.IndexFactory.of(index_ref)

## Preprocessing

todolist:
- tolower case
- punctuation removal
- stopwords
- stemming or lemmatization
- misspellings --> usually not preffered as its slow


In [ ]:
#performing preprocessing on the queries
import re
import nltk
from nltk.stem import PorterStemmer
from spellchecker import SpellChecker
from nltk.corpus import stopwords
from nltk import pos_tag

nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')

stemmer = PorterStemmer()
spell = SpellChecker()
stop_words = set(stopwords.words('english'))

def preprocess_text(text, level):
    if pd.isna(text):
        return ""
    
    # lowercasing
    text = text.lower()
    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # level 0: just lowercase + remove punctuation
    if level == 0:
        return text

    words = text.split()

    # level 1: POS filtering + stemming + stopword removal
    if level == 1:
        pos_tags = pos_tag(words)
        allowed_pos_tags = {'NN', 'NNS', 'VBN', 'JJ', 'RB', 'NNP', 'NNPS'}
        filtered_pos_words = [token for token, pos in pos_tags if pos in allowed_pos_tags]
        stemmed_words = [stemmer.stem(token) for token in filtered_pos_words]
        filtered_words = [token for token in stemmed_words if token not in stop_words]

    # level 2: spell correction + POS filtering + stemming + stopword removal
    if level == 2:
        corrected_words = [spell.correction(token) or token for token in words]
        pos_tags = pos_tag(corrected_words)
        allowed_pos_tags = {'NN', 'NNS', 'VBN', 'JJ', 'RB', 'NNP', 'NNPS'}
        filtered_pos_words = [token for token, pos in pos_tags if pos in allowed_pos_tags]
        stemmed_words = [stemmer.stem(token) for token in filtered_pos_words]
        filtered_words = [token for token in stemmed_words if token not in stop_words]

    return ' '.join(filtered_words)

print(verbosity_levels)

for level in verbosity_levels:
    print(f"\nPreprocessing with verbosity level: {level}")
    df_topics[f'preprocessed_title_{level}'] = df_topics['title'].apply(lambda x: preprocess_text(x, level))
    df_topics[f'preprocessed_description_{level}'] = df_topics['description'].apply(lambda x: preprocess_text(x, level))
    df_topics[f'preprocessed_narrative_{level}'] = df_topics['narrative'].apply(lambda x: preprocess_text(x, level))

print(df_topics.head(3))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nehiraltinkaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/nehiraltinkaya/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/nehiraltinkaya/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


[0, 1, 2]

Preprocessing with verbosity level: 0

Preprocessing with verbosity level: 1

Preprocessing with verbosity level: 2
                                     title  ...                           preprocessed_narrative_2
0                       coronavirus origin  ...  rang inform sarscov2 viru origin evolut anim s...
1  coronavirus response to weather changes  ...  rang inform sarscov2 viru viabil differ weathe...
2                     coronavirus immunity  ...  studi immun develop due infect sarscov2 cross ...

[3 rows x 12 columns]


In [36]:
#checking the effect of preprocessing on a sample query

sample = df_topics['title'][0]
print("Original:  ", sample)
print("Level 0:   ", preprocess_text(sample, 0))
print("Level 1:   ", preprocess_text(sample, 1))
print("Level 2:   ", preprocess_text(sample, 2))
print(preprocess_text("coronavyrus originn", 1))  # no spell correction
print(preprocess_text("coronavyrus originn", 2))  # spell correction applied first

Original:   coronavirus origin
Level 0:    coronavirus origin
Level 1:    coronaviru origin
Level 2:    coronaviru origin
coronavyru originn
coronavyru origin
